In [ ]:
# 1. Verify L4 GPU & Mount Google Drive inside the Colab IPython kernel
!nvidia-smi
from google.colab import drive
drive.mount('/content/drive', force_remount=False)


In [ ]:
# 2. Clone or update Lyrion repository and select L4 tier6b model profile
import os
if not os.path.exists('/content/lyrion/.git'):
    !git clone https://github.com/kingkong-cfl/lyrionalpha.git /content/lyrion
else:
    !git -C /content/lyrion pull --ff-only

#@title Lyrion L4 (tier6b 22GB VRAM) Model Profile Selector
# Select 'xl_turbo_4b' (fast 8-step 4B XL) or 'xl_sft_4b' (max studio quality 32-step CFG) for highest quality.
MODEL_PROFILE = "balanced_2b_4b_lm"  #@param ["fast_2b_1.7b", "balanced_2b_4b_lm", "xl_turbo_4b", "xl_sft_4b"]

PROFILE_CONFIGS = {
    "fast_2b_1.7b": ("acestep-v15-turbo", "acestep-5Hz-lm-1.7B", "false"),
    "balanced_2b_4b_lm": ("acestep-v15-turbo", "acestep-5Hz-lm-4B", "true"),
    "xl_turbo_4b": ("acestep-v15-xl-turbo", "acestep-5Hz-lm-4B", "true"),
    "xl_sft_4b": ("acestep-v15-xl-sft", "acestep-5Hz-lm-4B", "true"),
}
_dit, _lm, _offload = PROFILE_CONFIGS[MODEL_PROFILE]
os.environ["ACESTEP_CONFIG_PATH"] = _dit
os.environ["ACESTEP_LM_MODEL_PATH"] = _lm
os.environ["ACESTEP_OFFLOAD_TO_CPU"] = _offload
os.environ["ACESTEP_LM_BACKEND"] = "pt"
os.environ.setdefault("DEMUCS_MODEL", "htdemucs_ft")

# Optional: Set your Gemini API key for AI Producer lyric/tag planning
# os.environ["GEMINI_API_KEY"] = "your-api-key-here"
# Optional: Persist HuggingFace/Torch/ACE-Step weights to Google Drive to speed up restarts
# os.environ["LYRION_CACHE_MODELS_TO_DRIVE"] = "true"
print(f"[Lyrion Profile] {MODEL_PROFILE} -> DiT={_dit}, LM={_lm}, offload_to_cpu={_offload}")


In [ ]:
# 3. Install dependencies, launch Cloudflare tunnel, and start FastAPI backend
%cd /content/lyrion/colab-backend
%run scripts/colab_bootstrap.py
